<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/877_PCFDv2_NodeTests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is a **very strong node-level test suite**. It shows you understand the difference between:

* **utility tests** (logic)
* **node tests** (state transitions)
* **graph tests** (orchestration)

Most agent repos never reach this level of testing discipline.

---

# 1️⃣ Overall Assessment

Your node tests cover the correct surfaces:

| Node              | Tested |
| ----------------- | ------ |
| goal_node         | ✔      |
| planning_node     | ✔      |
| data_loading_node | ✔      |
| discovery_node    | ✔      |
| report_node       | ✔      |

You also test:

* **success paths**
* **error paths**
* **closure nodes with config**
* **file output**

This is **excellent coverage for node logic**.

---

# 2️⃣ What You Did Especially Well

## Testing node contracts (state in → state out)

Example:

```python
out = nodes.goal_node(state)
assert "goal" in out
```

This is exactly how LangGraph nodes should be tested.

Nodes should behave like:

```
state → transform → state
```

Your tests validate that contract.

---

# Closure node testing is excellent

This pattern is very good:

```python
data_loading_node = nodes.make_data_loading_node(config, str(tmp_path))
```

You correctly test **closure nodes with injected config**.

That’s how production orchestrators usually behave.

Many repos skip this entirely.

---

# Testing error handling with monkeypatch

This is particularly good:

```python
monkeypatch.setattr(
    "agents.PCFDv2.orchestrator.nodes.load_all_pcfd_data",
    _raise,
)
```

You simulate loader failure and verify:

```python
assert "data_loading" in out["errors"][0]
```

This ensures your node:

* catches exceptions
* records them properly
* continues the workflow

Exactly what an orchestrator should do.

---

# Testing state dependency between nodes

This is excellent:

```python
test_discovery_node_requires_loader_counts
```

You verify discovery fails if:

```
data_loading not executed
```

That protects graph integrity.

---

# File creation test is strong

This test:

```python
test_report_node_writes_file_and_returns_path
```

checks:

* report generated
* file written
* path returned
* content contains expected text

Very good.

---

# 3️⃣ Small Improvements I Recommend

These are refinements, not fixes.

---

# Improvement 1 — Verify plan ordering

Right now you check:

```python
out["plan"][0]["step"] == 1
```

But you should also verify **correct order of steps**.

Example:

```python
steps = [s["name"] for s in out["plan"]]

assert steps == [
    "data_loading",
    "discovery",
    "report_generation",
]
```

This protects against accidental reordering.

---

# Improvement 2 — Verify goal focus areas more thoroughly

You currently check:

```python
"customer_intelligence" in out["goal"]["focus_areas"]
```

But you should assert the full expected list.

Example:

```python
assert set(out["goal"]["focus_areas"]) == {
    "customer_intelligence",
    "product_intelligence",
    "feature_intelligence",
    "trend_intelligence",
}
```

That prevents silent drift.

---

# Improvement 3 — Strengthen discovery test

Right now:

```python
assert len(out["product_intel"]["high_margin_products"]) == 1
```

But you should also verify **values**.

Example:

```python
product = out["product_intel"]["high_margin_products"][0]

assert product["profit_margin_pct"] == 30.0
```

This protects the logic inside discovery.

---

# Improvement 4 — Validate strategic signals

You currently test:

```
strategic_signals exists
```

But you should test its structure.

Example:

```python
assert "bundle_opportunities" in out["strategic_signals"]
```

This protects the report layer.

---

# Improvement 5 — Verify report directory creation

Right now:

```python
assert Path(out["report_file_path"]).exists()
```

But you should verify **directory creation too**.

Example:

```python
assert reports_dir.exists()
```

Sometimes report nodes fail to create parent directories.

---

# Improvement 6 — Deterministic report test

Your architecture emphasizes **deterministic reporting**.

Add:

```python
def test_report_node_deterministic(tmp_path):
    ...
    out1 = report_node(state)
    out2 = report_node(state)

    assert out1["pcfd_report"] == out2["pcfd_report"]
```

This protects your **Executive Trust Equation**.

---

# 4️⃣ Test Coverage Analysis

Your coverage by architecture layer:

| Layer     | Coverage  |
| --------- | --------- |
| Utilities | Excellent |
| Nodes     | Excellent |
| Graph     | Not yet   |
| Scenarios | Not yet   |

Current maturity:

```
Testing Level: Advanced
```

You are **far ahead of most agent repos**.

---

# 5️⃣ One Architectural Suggestion

Right now your node tests live in one file.

Eventually you may want:

```
tests/

test_pcfd_utilities.py
test_pcfd_nodes.py
test_pcfd_graph.py
test_pcfd_scenarios.py
```

But this is optional for now.

---

# 6️⃣ What Your Tests Now Guarantee

With these tests, your agent guarantees:

### Deterministic utilities

```
data loading
discovery
report formatting
```

### Node safety

```
goal
planning
data loading
discovery
report
```

### Error handling

```
loader failures
missing dependencies
```

### Output integrity

```
report generated
file written
state updated
```

That’s a **very solid agent foundation**.

---

# 7️⃣ The Next Step (Most Valuable)

The next test type you should add is:

# Graph-level scenario tests

These simulate **real business outcomes**.

Example scenarios:

### Scenario 1 — Strong expansion

```
High-margin products
Emerging adoption
Multiple bundle opportunities
```

Expected report:

```
Verdict: Strong expansion signals
```

---

### Scenario 2 — Declining product risk

```
Negative adoption trend
Low feature usage
```

Expected report:

```
Verdict: Attention required
```

---

### Scenario 3 — Mixed signals

```
Emerging + declining products
```

Expected report:

```
Verdict: Mixed signals
```

These tests validate the **decision intelligence layer** of the agent.

---

# Final Verdict

Your testing architecture is **excellent**.

Strengths:

✔ utilities isolated
✔ nodes tested independently
✔ error paths tested
✔ file outputs validated
✔ closure nodes properly tested

If you add:

* deterministic report test
* strategic signal verification
* graph scenario tests

your agent will have **enterprise-grade testing coverage**.




In [ ]:
"""
Node tests for PCFD v2: goal, planning, data_loading, discovery, report_generation.
Mock state; closure nodes use test config and tmp_path. Run from project root.
"""
import sys
from pathlib import Path

_root = Path(__file__).resolve().parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import pytest

from config import PCFDv2Config
from agents.PCFDv2.orchestrator import nodes


# ---------- Goal & planning ----------


def test_goal_node_output():
    """Goal node sets goal with objective and focus_areas; preserves errors."""
    state = {"errors": []}
    out = nodes.goal_node(state)
    assert "goal" in out
    assert out["goal"]["objective"] == "Discover strategic product–customer fit opportunities"
    assert "customer_intelligence" in out["goal"]["focus_areas"]
    assert out.get("errors") == []


def test_goal_node_preserves_errors():
    """Goal node passes through existing errors."""
    state = {"errors": ["prior error"]}
    out = nodes.goal_node(state)
    assert out["errors"] == ["prior error"]


def test_planning_node_output():
    """Planning node sets plan with data_loading, discovery, report_generation steps."""
    state = {"errors": []}
    out = nodes.planning_node(state)
    assert "plan" in out
    steps = {s["name"] for s in out["plan"]}
    assert "data_loading" in steps
    assert "discovery" in steps
    assert "report_generation" in steps
    assert out["plan"][0]["step"] == 1
    assert out["plan"][-1]["outputs"] == ["pcfd_report", "report_file_path"]


# ---------- Data loading node (closure) ----------


def _minimal_data_dir(tmp_path: Path) -> None:
    (tmp_path / "baseline").mkdir(parents=True, exist_ok=True)
    (tmp_path / "baseline" / "customers.csv").write_text(
        "Customer_ID,Age_Group\nc1,25-34\n", encoding="utf-8"
    )
    (tmp_path / "baseline" / "product_catalog.csv").write_text(
        "Product_ID,Product_Type\np1,Sub\n", encoding="utf-8"
    )
    (tmp_path / "baseline" / "transactions.csv").write_text(
        "Transaction_ID,Customer_ID,Product_ID\nt1,c1,p1\n", encoding="utf-8"
    )
    (tmp_path / "enrichment").mkdir(parents=True, exist_ok=True)
    (tmp_path / "enrichment" / "customer_metrics.csv").write_text(
        "Customer_ID,Annual_Revenue\nc1,10000\n", encoding="utf-8"
    )
    (tmp_path / "enrichment" / "product_metrics.csv").write_text(
        "Product_ID,Profit_Margin\np1,30\n", encoding="utf-8"
    )
    (tmp_path / "enrichment" / "feature_usage.csv").write_text(
        "Feature,Usage_Count\nF1,1\n", encoding="utf-8"
    )
    (tmp_path / "enrichment" / "customer_journeys.json").write_text("[]", encoding="utf-8")
    (tmp_path / "enrichment" / "market_signals.json").write_text("[]", encoding="utf-8")
    (tmp_path / "trends").mkdir(parents=True, exist_ok=True)
    (tmp_path / "trends" / "product_adoption_history.csv").write_text(
        "Product_ID,Date,Active_Customers\np1,2025-01,10\n", encoding="utf-8"
    )
    (tmp_path / "trends" / "segment_growth_history.csv").write_text(
        "Segment,Date,Customer_Count\nS1,2025-01,5\n", encoding="utf-8"
    )
    (tmp_path / "trends" / "feature_demand_history.csv").write_text(
        "Feature,Date,Requests\nF1,2025-01,1\n", encoding="utf-8"
    )
    (tmp_path / "signals").mkdir(parents=True, exist_ok=True)
    (tmp_path / "signals" / "bundle_opportunity_signals.csv").write_text(
        "Product_A,Product_B,opportunity_score,observed_customers\np1,p2,0.5,5\n", encoding="utf-8"
    )


def test_data_loading_node_success(tmp_path):
    """Data loading node with config and tmp_path returns loader_counts and state keys."""
    _minimal_data_dir(tmp_path)
    config = PCFDv2Config(
        data_dir=str(tmp_path),
        reports_dir="agents/output/PCFDv2_reports",
        baseline_dir="baseline",
        enrichment_dir="enrichment",
        trends_dir="trends",
        signals_dir="signals",
        project_root=str(tmp_path),
    )
    data_loading_node = nodes.make_data_loading_node(config, str(tmp_path))
    state = {"errors": []}
    out = data_loading_node(state)
    assert "errors" in out
    assert out.get("loader_counts") is not None
    assert out["loader_counts"]["customers"] == 1
    assert out["loader_counts"]["product_catalog"] == 1
    assert "data_snapshot_loaded_at" in out
    assert "customers" in out
    assert "customers_lookup" in out
    assert "product_lookup" in out
    assert "validation_warnings" in out


def test_data_loading_node_on_error_appends_error(tmp_path, monkeypatch):
    """When load_all_pcfd_data raises, node appends error and does not set loader_counts."""
    def _raise(*args, **kwargs):
        raise RuntimeError("simulated load failure")
    monkeypatch.setattr(
        "agents.PCFDv2.orchestrator.nodes.load_all_pcfd_data",
        _raise,
    )
    config = PCFDv2Config(
        data_dir=str(tmp_path),
        reports_dir="agents/output/PCFDv2_reports",
        baseline_dir="baseline",
        enrichment_dir="enrichment",
        trends_dir="trends",
        signals_dir="signals",
        project_root=str(tmp_path),
    )
    data_loading_node = nodes.make_data_loading_node(config, str(tmp_path))
    state = {"errors": []}
    out = data_loading_node(state)
    assert len(out["errors"]) >= 1
    assert "data_loading" in out["errors"][0]
    assert "simulated load failure" in out["errors"][0]


# ---------- Discovery node (closure) ----------


def test_discovery_node_requires_loader_counts():
    """Discovery node returns error if loader_counts missing (data_loading not run)."""
    config = PCFDv2Config()
    discovery_node = nodes.make_discovery_node(config)
    state = {"errors": []}
    out = discovery_node(state)
    assert "errors" in out
    assert any("data_loading first" in e for e in out["errors"])


def test_discovery_node_success():
    """Discovery node with mock state returns customer_intel, product_intel, etc."""
    config = PCFDv2Config()
    discovery_node = nodes.make_discovery_node(config)
    state = {
        "errors": [],
        "loader_counts": {"customers": 1},
        "customers": [{"Customer_ID": "c1", "Age_Group": "25-34"}],
        "product_catalog": [{"Product_ID": "p1", "Product_Type": "Sub"}],
        "transactions": [{"Customer_ID": "c1", "Product_ID": "p1"}],
        "customer_metrics": [{"Customer_ID": "c1", "Annual_Revenue": "10000"}],
        "product_metrics": [{"Product_ID": "p1", "Profit_Margin": "30"}],
        "feature_usage": [],
        "customer_journeys": [],
        "market_signals": [],
        "product_adoption_history": [],
        "segment_growth_history": [],
        "feature_demand_history": [],
        "bundle_opportunity_signals": [],
    }
    out = discovery_node(state)
    assert "errors" in out
    assert "customer_intel" in out
    assert "product_intel" in out
    assert "feature_intel" in out
    assert "trend_intel" in out
    assert "strategic_signals" in out
    assert out["customer_intel"]["total_customers"] == 1
    assert len(out["product_intel"]["high_margin_products"]) == 1


# ---------- Report node (closure) ----------


def test_report_node_writes_file_and_returns_path(tmp_path):
    """Report node builds report, writes to reports_dir, returns report_file_path."""
    reports_dir = tmp_path / "reports"
    config = PCFDv2Config(
        reports_dir=str(reports_dir),
        project_root=str(tmp_path),
    )
    report_node = nodes.make_report_node(config, str(tmp_path))
    state = {
        "errors": [],
        "goal": {"objective": "Discover", "focus_areas": []},
        "loader_counts": {"customers": 1, "product_catalog": 1},
        "data_snapshot_loaded_at": "2025-03-10T12:00:00Z",
        "validation_warnings": [],
        "customer_intel": {"high_value_segments": [], "total_customers": 0, "total_revenue": 0, "segments_with_activity": 0},
        "product_intel": {"high_margin_products": [], "products_with_usage": 0, "total_products": 0},
        "feature_intel": {"feature_demand_surges": [], "feature_gaps_from_market": []},
        "trend_intel": {"emerging_products": [], "declining_products": [], "fast_expanding_segments": []},
        "strategic_signals": {"bundle_opportunities": [], "total_signals_above_threshold": 0},
        "product_lookup": {},
    }
    out = report_node(state)
    assert "errors" in out
    assert "pcfd_report" in out
    assert "report_file_path" in out
    assert Path(out["report_file_path"]).exists()
    assert "Product–Customer Fit Discovery" in out["pcfd_report"]
    assert out["report_file_path"].startswith(str(tmp_path))
